<a href="https://colab.research.google.com/github/VitorKng/engenharia-de-prompt-fundamentos/blob/main/Aula08_AutomacaoIA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#AUTOMAÇÃO EM IA
###AULA 8:______________________________
###NOME:______________________________

#MISSÃO 01
##Automação de Arquivos
Criar um programa que organize arquivos bagunçados em pastas automaticamente, baseando-se apenas na extensão do arquivo (ex: .pdf, .jpg). (Use apenas as biblioteca os ou shutil). Faça em arquivo de teste.


In [12]:
import os
import shutil

def organizar_arquivos(diretorio):
    # Entra no diretório especificado
    if not os.path.exists(diretorio):
        print("Diretório não encontrado.")
        return

    for arquivo in os.listdir(diretorio):
        caminho_completo = os.path.join(diretorio, arquivo)

        # Ignora pastas, foca apenas em arquivos
        if os.path.isfile(caminho_completo):
            # Extrai a extensão (ex: '.jpg') e remove o ponto
            extensao = os.path.splitext(arquivo)[1].lower().replace('.', '')

            # Se o arquivo não tiver extensão, joga para uma pasta 'outros'
            if not extensao:
                extensao = 'outros'

            # Define e cria a pasta de destino
            pasta_destino = os.path.join(diretorio, extensao)
            if not os.path.exists(pasta_destino):
                os.makedirs(pasta_destino)

            # Move o arquivo
            shutil.move(caminho_completo, os.path.join(pasta_destino, arquivo))
            print(f"Movido: {arquivo} -> {extensao}/")

# Teste o script em uma pasta específica
if __name__ == "__main__":
    # Substitua pelo caminho da sua pasta de testes
    pasta_teste = "./meus_arquivos_baguncados"

    # Criando pasta de teste e arquivos fictícios para demonstração
    if not os.path.exists(pasta_teste):
        os.makedirs(pasta_teste)
        open(f"{pasta_teste}/documento.pdf", 'a').close()
        open(f"{pasta_teste}/foto.jpg", 'a').close()
        open(f"{pasta_teste}/planilha.xlsx", 'a').close()
        open(f"{pasta_teste}/notas.txt", 'a').close()

    organizar_arquivos(pasta_teste)
    print("\nOrganização concluída!")



Movido: planilha.xlsx -> xlsx/
Movido: documento.pdf -> pdf/
Movido: notas.txt -> txt/
Movido: foto.jpg -> jpg/

Organização concluída!


#MISSÃO 02

##Consulta Simples a APIs

''Desenvolver um script que consulte uma API pública (via CEP) e retorne as informações de endereço formatadade maneira legível.''

crie e desenvolva um sistema de API pública (via CEP) e retorne as informações formatadas de maneira legível, crie com uma exceção de erro.

In [32]:
import requests

class CEPNotFoundError(Exception):
    """Exceção lançada quando o CEP é válido em formato, mas não existe na base de dados."""
    pass

def buscar_endereco(cep):
    # Limpa o CEP (remove pontos e hífens)
    cep_limpo = "".join(filter(str.isdigit, cep))

    try:
        # Verifica se o formato é válido antes de chamar a API
        if len(cep_limpo) != 8:
            raise ValueError("O CEP deve conter exatamente 8 dígitos numéricos.")

        url = f"https://viacep.com.br{cep_limpo}/json/"
        response = requests.get(url, timeout=5)

        # Lança erro se a conexão falhar (ex: 404 ou 500)
        response.raise_for_status()

        dados = response.json()

        # A ViaCEP retorna {'erro': True} para CEPs inexistentes
        if "erro" in dados:
            raise CEPNotFoundError(f"O CEP '{cep}' não foi encontrado na base de dados.")

        # Retorno formatado
        print(f"\n✅ RESULTADO PARA: {dados['cep']}")
        print("-" * 30)
        print(f"Rua:      {dados.get('logradouro', 'N/A')}")
        print(f"Bairro:   {dados.get('bairro', 'N/A')}")
        print(f"Cidade:   {dados.get('localidade')} - {dados.get('uf')}")
        print(f"DDD:      {dados.get('ddd')}")
        print("-" * 30)

    except ValueError as e:
        print(f"❌ ERRO DE VALIDAÇÃO: {e}")
    except CEPNotFoundError as e:
        print(f"❌ ERRO DE BUSCA: {e}")
    except requests.exceptions.RequestException as e:
        print(f"❌ ERRO DE CONEXÃO: Não foi possível acessar o servidor. {e}")
    except Exception as e:
        print(f"❌ ERRO INESPERADO: {e}")

# Teste do sistema
if __name__ == "__main__":
    entrada = input("Digite o CEP (ex: 01001-000): ")
    buscar_endereco(entrada)


Digite o CEP (ex: 01001-000): 73150150
❌ ERRO DE CONEXÃO: Não foi possível acessar o servidor. HTTPSConnectionPool(host='viacep.com.br73150150', port=443): Max retries exceeded with url: /json/ (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7e0b5ae00290>: Failed to resolve 'viacep.com.br73150150' ([Errno -2] Name or service not known)"))


#MISSÃO 03

##Sistema de Notificações

Crie um programa autonomo que dispare alertas (mensagem de alerta no sistema ou e-mail simulado) unicamente quando uma condição critica for atingida.

In [ ]:
import psutil
import time
import os
from datetime import datetime

# Configurações
LIMITE_CRITICO = 80.0  # Porcentagem de uso da CPU
INTERVALO_CHECAGEM = 5  # Segundos entre cada verificação

def disparar_alerta(valor_atual):
    horario = datetime.now().strftime("%H:%M:%S")
    mensagem = f"ALERTA CRÍTICO: Uso de CPU em {valor_atual}% às {horario}!"

    # 1. Alerta de Sistema (Visual)
    # No Windows usa msg, no Linux/Mac tenta usar notify-send
    if os.name == "nt":
        os.system(f'msg * "{mensagem}"')
    else:
        os.system(f'echo "{mensagem}"')

    # 2. Simulação de E-mail
    print(f"\n[E-MAIL ENVIADO PARA SUPORTE]")
    print(f"Assunto: Incidente Detectado")
    print(f"Corpo: {mensagem}\n")

def monitorar_sistema():
    print(f"--- Monitoramento Iniciado (Limite: {LIMITE_CRITICO}%) ---")

    try:
        while True:
            # Captura o uso da CPU
            uso_cpu = psutil.cpu_percent(interval=1)

            if uso_cpu > LIMITE_CRITICO:
                disparar_alerta(uso_cpu)
                # Pausa maior após alerta para evitar spam de mensagens
                time.sleep(60)

            time.sleep(INTERVALO_CHECAGEM)

    except KeyboardInterrupt:
        print("\nMonitoramento encerrado pelo usuário.")

if __name__ == "__main__":
    monitorar_sistema()


--- Monitoramento Iniciado (Limite: 80.0%) ---
